<a href="https://colab.research.google.com/github/ragiokay/ML-2026-HW7/blob/main/ML2026_Spring_HW7_Model_Merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ML2026 Spring HW7 — Model Merging**

在這份作業中，將實作 **Model Merging** 。

我們會使用兩個 7B 模型：
- `augmxnt/shisa-gamma-7b-v1`（擅長日文）
- `WizardLMTeam/WizardMath-7B-V1.1`（擅長數學推理）

**❗重要規則：**

- **請勿使用上述兩個模型以外的任何其他模型。**
- 你可以自由使用任何套件（不限於 mergekit）或自己撰寫的演算法來進行 merging。
- **Merging 的嚴格定義：合併後的模型總參數量必須與單一 base model 相同。** 換言之，MoE（Mixture of Experts）、Ensemble、Stacking 等會使 inference 時可使用的參數量增加的做法，皆視為違規。

你的目標是：
1. 先觀察兩個 base model 各自的能力
2. 嘗試各種 merge 方法，把兩個模型合併
3. 在日文數學題上測試合併後的模型表現，並將結果上傳到 judgeboi

# **Section 0: Preparing**

## Mount your notebook to your Drive

In [7]:
!git pull origin main

fatal: not a git repository (or any of the parent directories): .git


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Activate GPU
Model Merging 與 Inference 都需要 GPU，請務必啟用。

### **MUST READ**:

Colab does **NOT** guarantee the GPU access for free user ([ref](https://research.google.com/colaboratory/faq.html#idle-timeouts)). It is possible you get a message saying "Cannot connect to GPU backend" which means there are no enough GPU resources for you now. When this happens, you may need to **wait for one (or more) day or login different Google account to do the homework**.

### Enable GPU

1. Click on "Runtime" (or "執行階段") in the header.
2. Click on "Change runtime type" (or "變更執行階段類型") in the drop menu.
3. Select "T4 GPU" and save. (You can select "A100 GPU" or "V100 GPU" if you have Colab Pro)

## Check the status of your GPU

In [9]:
!nvidia-smi

Thu May 21 01:35:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install Packages & Download Data

安裝 `mergekit`（模型合併工具）及下載日文數學題目檔案。

In [10]:
# Install mergekit
!pip install -q mergekit

# Install / upgrade gdown for Google Drive download
!pip install -U -q gdown

# Download the Japanese math question file
!gdown 1_9OAxDwQgz1jUnlhHlkk3lLCTpY1Ib8Q -O /content/jp_math_question.json
# Verify download
import json
with open("/content/jp_math_question.json", "r") as f:
    questions = json.load(f)
print(f"成功載入 {len(questions)} 題日文數學題目！")
print(f"範例題目: {questions[0]['question'][:80]}...")
print(f"範例答案: {questions[0]['answer_number']}")

Downloading...
From: https://drive.google.com/uc?id=1_9OAxDwQgz1jUnlhHlkk3lLCTpY1Ib8Q
To: /content/jp_math_question.json
100% 4.68k/4.68k [00:00<00:00, 21.4MB/s]
成功載入 20 題日文數學題目！
範例題目: 太郎は平成5年に生まれました。次郎は令和2年に生まれました。太郎は次郎より何歳年上ですか？...
範例答案: 27


# **Section 1: Explore Base Models — 觀察兩個 Base Model 的能力**
## 若時間不足可以先跳到Section 2

在合併之前，我們先分別載入兩個 base model，觀察它們各自的強項與弱項。

- **shisa-gamma-7b-v1**：以日文能力見長的模型
- **WizardMath-7B-V1.1**：以數學推理見長的模型

我們會各丟一個 **日文問題** 和一個 **數學問題** 給兩個模型，直覺感受它們的差異。

## 1.1 定義推論用的 helper function

建立通用的推論與評估函數，後續兩個模型都會用到。

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """通用推論函數：給定模型與 prompt，回傳生成結果"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output[len(prompt):].strip()

In [ ]:
import re

############################################################
# Prompt 模板
USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

# ⛔ 嚴禁修改 USER_PROMPT_TEMPLATE！
#
# 本作業聚焦於 Model Merging 技術本身，
# 而非 Prompt Engineering，因此為確保所有同學的
# 評測條件一致，無論任何情況皆請勿修改以下 prompt。
# 違反此規則將視為違規。
############################################################


def extract_number(text):
    """從模型輸出中提取數字答案"""
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

def evaluate_model(model, tokenizer, questions, model_name):
    """在日文數學題集上評估模型"""
    correct = 0
    results = []
    print(f"\n{'='*60}")
    print(f"📊 評估模型: {model_name}")
    print(f"{'='*60}\n")

    for i, item in enumerate(questions):
        prompt = USER_PROMPT_TEMPLATE.format(question=item['question'])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                repetition_penalty=1.1
            )
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer_part = full_output[len(prompt):].strip()
        predicted = extract_number(answer_part)
        expected = item["answer_number"]
        is_correct = predicted == expected
        if is_correct:
            correct += 1
        results.append({
            "id": i + 1, "correct": is_correct,
            "predicted": predicted, "expected": expected,
            "raw": answer_part,
        })
        mark = "✅" if is_correct else "❌"
        print(f"{mark} Q{i+1:02d} | 預測: {str(predicted):>6s} | 正解: {expected:>6s}")
        print(f"     | 題目: {item['question'][:60]}")
        print(f"     | raw: {answer_part[:120]}")
        print()

    print(f"{'='*50}")
    print(f"🎯 {model_name} 正確率: {correct}/{len(questions)} ({correct/len(questions)*100:.1f}%)")
    print(f"{'='*50}")
    return correct, results

## 1.2 測試 Base Model ①：shisa-gamma-7b-v1（日文模型）

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("載入 shisa-gamma-7b-v1 ...")
shisa_name = "augmxnt/shisa-gamma-7b-v1"
shisa_tokenizer = AutoTokenizer.from_pretrained(shisa_name)
shisa_model = AutoModelForCausalLM.from_pretrained(
    shisa_name, torch_dtype=torch.float16, device_map="auto"
)
print("shisa-gamma-7b-v1 載入完成！")

### 日文能力測試（shisa）

In [ ]:
jp_prompt = "日本の四季の特徴について、それぞれ簡潔に説明してください。"

print("=" * 60)
print("📝 日文 Prompt:", jp_prompt)
print("=" * 60)
print("\n🔵 【shisa-gamma-7b（日文模型）】的回答：")
print("-" * 40)
shisa_jp_ans = generate_response(shisa_model, shisa_tokenizer, jp_prompt)
print(shisa_jp_ans[:500])

### 數學能力測試（shisa）

In [ ]:
math_prompt = (
    "Solve the following math problem step by step.\n\n"
    "Problem: A store sells apples for $2 each and oranges for $3 each. "
    "If a customer buys 5 apples and 4 oranges, and pays with a $50 bill, "
    "how much change should the customer receive?\n\n"
    "Solution:"
)

print("=" * 60)
print("📝 Math Prompt:")
print(math_prompt)
print("=" * 60)
print("\n🔵 【shisa-gamma-7b（日文模型）】的回答：")
print("-" * 40)
shisa_math_ans = generate_response(shisa_model, shisa_tokenizer, math_prompt)
print(shisa_math_ans[:500])

print("\n" + "=" * 60)
print("💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28")
print("=" * 60)

### 在日文數學題集上評估（shisa）

In [ ]:
with open("/content/jp_math_question.json", "r") as f:
    questions = json.load(f)

shisa_correct, shisa_results = evaluate_model(
    shisa_model, shisa_tokenizer, questions, "shisa-gamma-7b（日文模型）"
)

### 釋放 shisa 模型記憶體

In [ ]:
import gc

del shisa_model, shisa_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("已釋放 shisa-gamma-7b 記憶體！")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

## 1.3 測試 Base Model ②：WizardMath-7B-V1.1（數學模型）

In [ ]:
print("載入 WizardMath-7B-V1.1 ...")
wizard_name = "WizardLMTeam/WizardMath-7B-V1.1"
wizard_tokenizer = AutoTokenizer.from_pretrained(wizard_name)
wizard_model = AutoModelForCausalLM.from_pretrained(
    wizard_name, torch_dtype=torch.float16, device_map="auto"
)
print("WizardMath-7B-V1.1 載入完成！")

### 日文能力測試（WizardMath）

In [ ]:
print("=" * 60)
print("📝 日文 Prompt:", jp_prompt)
print("=" * 60)
print("\n🟠 【WizardMath-7B（數學模型）】的回答：")
print("-" * 40)
wizard_jp_ans = generate_response(wizard_model, wizard_tokenizer, jp_prompt)
print(wizard_jp_ans[:500])

### 數學能力測試（WizardMath）

In [ ]:
print("=" * 60)
print("📝 Math Prompt:")
print(math_prompt)
print("=" * 60)
print("\n🟠 【WizardMath-7B（數學模型）】的回答：")
print("-" * 40)
wizard_math_ans = generate_response(wizard_model, wizard_tokenizer, math_prompt)
print(wizard_math_ans[:500])

print("\n" + "=" * 60)
print("💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28")
print("=" * 60)

### 在日文數學題集上評估（WizardMath）

In [ ]:
wizard_correct, wizard_results = evaluate_model(
    wizard_model, wizard_tokenizer, questions, "WizardMath-7B（數學模型）"
)

In [ ]:
del wizard_model, wizard_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("已釋放 WizardMath-7B 記憶體！")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

### 🔍 Base Model Baseline

記下兩個 base model 的正確率，之後合併模型時可以做比較：
- shisa-gamma-7b：擅長日文，但數學推理能力可能不足
- WizardMath-7B：擅長數學推理，但可能無法理解日文題意

理想的 merge 應該要能結合兩者的優勢！

# **Section 2: Model Merging — 合併模型**

## 2.1 撰寫 Merge 設定檔

以下是一個最簡單的 **50/50 Linear Merge**（線性插值合併）設定。

### ⚠️ 這是你的主要修改區域！

你可以嘗試不同的合併策略，例如：
- **linear**：最直覺的加權平均（調整 `weight` 比例）
- **slerp**：球面線性插值（調整 `t` 參數）
- **ties**：TIES-Merging（需設定 `density` 與 `weight`）
- **dare_ties**：DARE + TIES（需設定 `density`、`weight`）

請參考 [mergekit 文件](https://github.com/arcee-ai/mergekit?tab=readme-ov-file#merge-methods) 了解更多合併方法與參數。

In [ ]:
############################################################
# ⚠️ 【可修改區域 — TODO】
#
# ═══════════════════════════════════════════════════════════
# 最終方法：7 個合併模型 Ensemble Majority Voting
# 整體正確率：90%（20 題中 18 題）
# ═══════════════════════════════════════════════════════════
#
# 【實驗過程概述】
#
# Round 1 — DARE_TIES（base: Mistral-7B-v0.1），15 組參數組合
#   最佳 55%。rescale=True 導致 L1 norm 爆炸，改 rescale:false
#   後仍卡在 55%；shisa 語言能力被大量稀釋。
#
# Round 2 — 換 base 為 shisa，測試 dare_ties/ties/dare_linear/
#   linear/slerp/task_arithmetic 共 18 組
#   突破：linear merge（shisa=0.6, wizard=0.4）直接跳到 80%。
#   layer-gradient 一律比 flat weight 差。
#
# Round 3 — 圍繞 linear 0.6/0.4 精細搜索（17 組）
#   結論：80% 是瓶頸。Q3（俳句音節=255）與 Q4（硬幣組合=666）
#   是知識盲點，任何 linear 組合皆無法答對。
#
# Round 4A — Ensemble Voting（5 個 80% config 多數決）
#   T1(task_arith w=0.4) ≡ L1(linear 0.6/0.4) 數學等價；
#   T2(task_arith w=0.35) ≡ L6(linear 0.65/0.35) 數學等價。
#   5 票實為 3 個獨立模型；Q8/Q11 恰好 2:2 平手，tie-break 選錯 → 80%
#
# Round 4B — DFS Frankenmerge（passthrough 按 layer 拼接）
#   DFS2(shisa前16層+wizard後16層)=65%
#   DFS3(linear_6040前24層+shisa後8層)=60%
#   DFS3 是唯一答對 Q3=255 的配置，但整體弱於 linear。
#
# Round 5 — 7 票 Ensemble + L6 優先 tie-break
#   Q8：L6+T2+DFS2 各自獨立給出 6 → 3:2 真實多數決 ✅
#   Q11：7 vs 8 各 3 票，L6 優先 tie-break → 7 ✅
#   Q3/Q4 仍是知識盲點。最終 18/20 = 90%
#
# 規則確認：每個合併模型參數量 = 7B（等於單一 base model）；
# 推論時 GPU 上每次只載入一個模型，符合作業規定。
############################################################

import os

SHISA  = "augmxnt/shisa-gamma-7b-v1"
WIZARD = "WizardLMTeam/WizardMath-7B-V1.1"
PS_MERGED_DIR = "./ps_merged"  # linear_6040，供 DFS3 使用

# Tie-break 優先順序：L6 排第一
# 依序 merge + infer，最後 majority vote
ENSEMBLE_CONFIGS = [
    {
        "name": "L6_linear_6535",
        "desc": "[linear] shisa=0.65, wizard=0.35",
        "yaml": f"models:\n  - model: {SHISA}\n    parameters:\n      weight:\n        - value: 0.65\n  - model: {WIZARD}\n    parameters:\n      weight:\n        - value: 0.35\nmerge_method: linear\ndtype: float16",
    },
    {
        "name": "T2_taskarith_w035",
        "desc": "[task_arithmetic] w=0.35",
        "yaml": f"models:\n  - model: {WIZARD}\n    parameters:\n      weight: 0.35\nmerge_method: task_arithmetic\nbase_model: {SHISA}\ndtype: float16",
    },
    {
        "name": "L1_linear_6040",
        "desc": "[linear] shisa=0.6, wizard=0.4",
        "yaml": f"models:\n  - model: {SHISA}\n    parameters:\n      weight:\n        - value: 0.6\n  - model: {WIZARD}\n    parameters:\n      weight:\n        - value: 0.4\nmerge_method: linear\ndtype: float16",
    },
    {
        "name": "T1_taskarith_w04",
        "desc": "[task_arithmetic] w=0.4",
        "yaml": f"models:\n  - model: {WIZARD}\n    parameters:\n      weight: 0.4\nmerge_method: task_arithmetic\nbase_model: {SHISA}\ndtype: float16",
    },
    {
        "name": "L2_linear_5842",
        "desc": "[linear] shisa=0.58, wizard=0.42",
        "yaml": f"models:\n  - model: {SHISA}\n    parameters:\n      weight:\n        - value: 0.58\n  - model: {WIZARD}\n    parameters:\n      weight:\n        - value: 0.42\nmerge_method: linear\ndtype: float16",
    },
    {
        "name": "DFS2_shisa16_wizard16",
        "desc": "[passthrough] shisa 前16層 + wizard 後16層",
        "yaml": f"slices:\n  - sources:\n      - model: {SHISA}\n        layer_range: [0, 16]\n  - sources:\n      - model: {WIZARD}\n        layer_range: [16, 32]\nmerge_method: passthrough\ntokenizer_source: union\ndtype: float16",
    },
    {
        "name": "DFS3_psmerged24_shisa8",
        "desc": "[passthrough] linear_6040 前24層 + shisa 後8層",
        "yaml": f"slices:\n  - sources:\n      - model: {PS_MERGED_DIR}\n        layer_range: [0, 24]\n  - sources:\n      - model: {SHISA}\n        layer_range: [24, 32]\nmerge_method: passthrough\ntokenizer_source: union\ndtype: float16",
    },
]

# 將第一個 config（L6）寫入 merge_config.yml 供備存
with open("merge_config.yml", "w") as f:
    f.write(ENSEMBLE_CONFIGS[0]["yaml"])
print(f"✅ 共 {len(ENSEMBLE_CONFIGS)} 個 ensemble config 定義完成")
for c in ENSEMBLE_CONFIGS:
    print(f"   {c['name']}: {c['desc']}")


In [ ]:
# 備存 merge_config.yml 到 Google Drive
import shutil, os
drive_backup_dir = "/content/drive/MyDrive/ML2026/HW7"
os.makedirs(drive_backup_dir, exist_ok=True)
shutil.copy("merge_config.yml", os.path.join(drive_backup_dir, "merge_config.yml"))
print(f"✅ merge_config.yml 已備存至 {drive_backup_dir}/merge_config.yml")

## 2.2 執行合併

Ensemble 中所有模型的合併與推論都整合在 Section 3.2 完成。
此處的 merge / load 步驟（2.2 / 2.3）可略過，直接跳到 Section 3。


In [ ]:
import yaml
import shutil
import os
from mergekit.config import MergeConfiguration
from mergekit.merge import run_merge
from mergekit.options import MergeOptions

# 清除舊的合併模型
if os.path.exists("./merged_model"):
    shutil.rmtree("./merged_model")

# 讀取設定檔
with open("merge_config.yml", "r") as f:
    merge_config = MergeConfiguration.model_validate(yaml.safe_load(f))

# 執行合併（random_seed=42 讓 DARE 隨機剪枝結果可重現）
run_merge(
    merge_config,
    out_path="./merged_model",
    options=MergeOptions(
        cuda=True,
        lazy_unpickle=True,
        allow_crimes=True,
        random_seed=42,
    ),
)
print("合併完成！")

## 2.3 確認合併輸出

In [ ]:
import os

output_dir = "./merged_model"
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    total_size = sum(os.path.getsize(os.path.join(output_dir, f)) for f in files)
    print(f"\n✅ 合併完成！檔案數: {len(files)}, 總大小: {total_size / 1e9:.2f} GB")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f}: {size / 1e6:.1f} MB")
else:
    print("❌ 合併失敗，請檢查上方錯誤訊息")

# **Section 3: Inference — 測試合併後的模型**

## 3.1 載入合併後的模型

In [ ]:
# Section 3 的模型載入由 3.2 的 ensemble 迴圈統一處理
# 如需單獨預覽主模型輸出，可先執行 Section 2.2 的合併，再執行本格
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re
print("✅ 套件已載入，請直接執行 3.2 的 Ensemble Inference")


## 3.2 在日文數學題上測試合併模型

使用與 Section 1 相同的題目與評估方式。
Inference 時間可能會需要1-2小時，請耐心等待

In [ ]:
# ================================================================
# Section 3.2 — Ensemble Inference + Majority Voting
# ================================================================
# 依序執行 7 個合併模型推論，每次只有一個 7B 模型在 GPU 上，
# 最後以多數決（L6 優先 tie-break）決定每題答案。

import yaml, json, shutil, os, gc, time
import torch
from collections import Counter
from mergekit.config import MergeConfiguration
from mergekit.merge import run_merge
from mergekit.options import MergeOptions
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

MERGED_DIR    = "./merged_model"
QUESTION_FILE = "/content/jp_math_question.json"

USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

def extract_number(text):
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

def majority_vote(votes_by_name, priority_order):
    valid = [v for v in votes_by_name.values() if v is not None]
    if not valid:
        return None
    counter = Counter(valid)
    max_count = counter.most_common(1)[0][1]
    if max_count < 2:
        for name in priority_order:
            if votes_by_name.get(name) is not None:
                return votes_by_name[name]
        return None
    top = {ans for ans, cnt in counter.items() if cnt == max_count}
    if len(top) == 1:
        return top.pop()
    for name in priority_order:
        if votes_by_name.get(name) in top:
            return votes_by_name[name]
    return counter.most_common(1)[0][0]

with open(QUESTION_FILE, "r", encoding="utf-8") as f:
    questions = json.load(f)

# ── Step 0：為 DFS3 建立 ps_merged（linear_6040 base）────────
if not os.path.exists(PS_MERGED_DIR):
    print("🔧 建立 ps_merged (linear_6040) 供 DFS3 使用...")
    ps_yaml = f"""models:
  - model: {SHISA}
    parameters:
      weight:
        - value: 0.6
  - model: {WIZARD}
    parameters:
      weight:
        - value: 0.4
merge_method: linear
dtype: float16"""
    ps_cfg = MergeConfiguration.model_validate(yaml.safe_load(ps_yaml))
    run_merge(ps_cfg, out_path=PS_MERGED_DIR,
              options=MergeOptions(cuda=True, lazy_unpickle=True, allow_crimes=True))
    print(f"✅ ps_merged 完成")
else:
    print(f"✅ ps_merged 已存在，跳過")

# ── Step 1：逐一 merge + infer，收集答案 ─────────────────────
priority_order = [c["name"] for c in ENSEMBLE_CONFIGS]
answers_matrix = {}

for cfg in ENSEMBLE_CONFIGS:
    print(f"\n{'='*55}\n🚀 {cfg['name']}: {cfg['desc']}\n{'='*55}")
    t0 = time.time()

    if os.path.exists(MERGED_DIR):
        shutil.rmtree(MERGED_DIR)

    merge_cfg = MergeConfiguration.model_validate(yaml.safe_load(cfg["yaml"]))
    run_merge(merge_cfg, out_path=MERGED_DIR,
              options=MergeOptions(cuda=True, lazy_unpickle=True,
                                   allow_crimes=True, random_seed=42))
    print(f"✅ Merge: {(time.time()-t0)/60:.1f} min")

    tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR, torch_dtype=torch.float16, device_map="auto")

    answers = []
    for i, item in enumerate(questions):
        prompt = USER_PROMPT_TEMPLATE.format(question=item["question"])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512,
                                     do_sample=False, repetition_penalty=1.1)
        answer_part = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()
        predicted = extract_number(answer_part)
        answers.append(predicted)
        mark = "✅" if predicted == item["answer_number"] else "❌"
        print(f"  {mark} Q{i+1:02d} | pred={str(predicted):>8s} | exp={item['answer_number']}")

    answers_matrix[cfg["name"]] = answers
    single_acc = sum(1 for a, q in zip(answers, questions) if a == q["answer_number"])
    print(f"  → 單獨正確率: {single_acc}/20 = {single_acc/20*100:.0f}%  ({(time.time()-t0)/60:.1f} min)")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# ── Step 2：Majority Voting ───────────────────────────────────
print(f"\n{'='*55}\n📊 Majority Vote 結果\n{'='*55}")
results = []
for i, item in enumerate(questions):
    votes = {name: answers_matrix[name][i] for name in priority_order}
    winner = majority_vote(votes, priority_order)
    is_correct = winner == item["answer_number"]
    results.append({"id": i+1, "winner": winner, "correct": is_correct,
                    "expected": item["answer_number"], "raw": str(winner) if winner else ""})
    mark = "✅" if is_correct else "❌"
    print(f"  {mark} Q{i+1:02d} | voted={str(winner):>8s} | exp={item['answer_number']}")

correct = sum(1 for r in results if r["correct"])
print(f"\n🏆 Ensemble 正確率: {correct}/20 = {correct/20*100:.0f}%")

# ── Step 3：儲存 submission.json ──────────────────────────────
submission_path = "/content/submission.json"
submission_data = [{"id": r["id"], "raw": r["raw"]} for r in results]
with open(submission_path, "w", encoding="utf-8") as f:
    json.dump(submission_data, f, ensure_ascii=False, indent=2)
print(f"\n📁 submission.json 已儲存至 {submission_path}")


In [ ]:
# 備存 submission.json 到 Google Drive
import shutil, os
drive_backup_dir = "/content/drive/MyDrive/ML2026/HW7"
os.makedirs(drive_backup_dir, exist_ok=True)
shutil.copy(submission_path, os.path.join(drive_backup_dir, "submission.json"))
print(f"✅ submission.json 已備存至 {drive_backup_dir}/submission.json")

# **Section 4: Grid Search — Linear 0.56~0.64 精細掃描**

圍繞 80% 甜蜜點掃描 6 組尚未測試（或重驗）的 linear 配比，
觀察各組的逐題答案是否不同，尋找可能突破 80% 的配置。

結果存至 `/content/drive/MyDrive/ML2026/HW7/grid_search_results_r5.jsonl`

In [ ]:
# ================================================================
# Section 4.1 — Round 5 configs：精細掃描 0.56~0.64
# ================================================================
# 已知：0.58/0.42 = 75%，0.60/0.40 = 80%，0.65/0.35 = 80%
# 目的：找出 0.56~0.64 之間逐題答案不同的組合
# Note: 0.62/0.38 曾在 Round 3 測試，此處重驗以確認錯題集合

SHISA  = "augmxnt/shisa-gamma-7b-v1"
WIZARD = "WizardLMTeam/WizardMath-7B-V1.1"

def linear_flat(sw, ww):
    return f"""models:
  - model: {SHISA}
    parameters:
      weight:
        - value: {sw}
  - model: {WIZARD}
    parameters:
      weight:
        - value: {ww}
merge_method: linear
dtype: float16"""

SEARCH_CONFIGS_R5 = [
    {"name": "R5_linear_5644", "desc": "[linear] shisa=0.56, wizard=0.44", "yaml": linear_flat(0.56, 0.44)},
    {"name": "R5_linear_5743", "desc": "[linear] shisa=0.57, wizard=0.43", "yaml": linear_flat(0.57, 0.43)},
    {"name": "R5_linear_6139", "desc": "[linear] shisa=0.61, wizard=0.39", "yaml": linear_flat(0.61, 0.39)},
    {"name": "R5_linear_6238", "desc": "[linear] shisa=0.62, wizard=0.38", "yaml": linear_flat(0.62, 0.38)},
    {"name": "R5_linear_6337", "desc": "[linear] shisa=0.63, wizard=0.37", "yaml": linear_flat(0.63, 0.37)},
    {"name": "R5_linear_6436", "desc": "[linear] shisa=0.64, wizard=0.36", "yaml": linear_flat(0.64, 0.36)},
]

print(f"✅ 共 {len(SEARCH_CONFIGS_R5)} 組 Round 5 config")
for c in SEARCH_CONFIGS_R5:
    print(f"  {c['name']}: {c['desc']}")


In [ ]:
# ================================================================
# Section 4.2 — Grid Search 主循環（Crash 可重跑，自動跳過已完成）
# ================================================================
import yaml, json, shutil, os, gc, time, datetime
import torch
from mergekit.config import MergeConfiguration
from mergekit.merge import run_merge
from mergekit.options import MergeOptions
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

RESULTS_FILE  = "/content/drive/MyDrive/ML2026/HW7/grid_search_results_r5.jsonl"
MERGED_DIR    = "./merged_model"
QUESTION_FILE = "/content/jp_math_question.json"

USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

def extract_number(text):
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

with open(QUESTION_FILE, "r", encoding="utf-8") as f:
    questions = json.load(f)

# 跳過已完成
done_names = set()
os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        for line in f:
            try:
                r = json.loads(line)
                done_names.add(r["name"])
            except:
                pass
    if done_names:
        print(f"⏭️  已完成，跳過: {sorted(done_names)}")

total = len(SEARCH_CONFIGS_R5)
for idx, cfg in enumerate(SEARCH_CONFIGS_R5, 1):
    if cfg["name"] in done_names:
        print(f"[{idx}/{total}] ⏭️  Skip {cfg['name']}")
        continue

    print(f"\n{'='*60}")
    print(f"[{idx}/{total}] 🚀 {cfg['name']}: {cfg['desc']}")
    print(f"{'='*60}")
    t_start = time.time()

    if os.path.exists(MERGED_DIR):
        shutil.rmtree(MERGED_DIR)

    try:
        merge_cfg = MergeConfiguration.model_validate(yaml.safe_load(cfg["yaml"]))
        run_merge(merge_cfg, out_path=MERGED_DIR,
                  options=MergeOptions(cuda=True, lazy_unpickle=True,
                                       allow_crimes=True, random_seed=42))
        t_merge = time.time() - t_start
        print(f"✅ Merge: {t_merge/60:.1f} min")
    except Exception as e:
        print(f"❌ Merge failed: {e}")
        with open(RESULTS_FILE, "a") as f:
            f.write(json.dumps({"name": cfg["name"], "desc": cfg["desc"],
                                "error": str(e), "stage": "merge",
                                "timestamp": datetime.datetime.now().isoformat()},
                               ensure_ascii=False) + "\n")
        continue

    try:
        tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
        model = AutoModelForCausalLM.from_pretrained(
            MERGED_DIR, torch_dtype=torch.float16, device_map="auto")
    except Exception as e:
        print(f"❌ Load failed: {e}")
        with open(RESULTS_FILE, "a") as f:
            f.write(json.dumps({"name": cfg["name"], "desc": cfg["desc"],
                                "error": str(e), "stage": "load",
                                "timestamp": datetime.datetime.now().isoformat()},
                               ensure_ascii=False) + "\n")
        continue

    correct = 0
    detail = []
    for i, item in enumerate(questions):
        try:
            prompt = USER_PROMPT_TEMPLATE.format(question=item["question"])
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=512,
                                         do_sample=False, repetition_penalty=1.1)
            answer_part = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()
            predicted = extract_number(answer_part)
            expected = item["answer_number"]
            is_correct = predicted == expected
            if is_correct:
                correct += 1
            detail.append({"q": i+1, "correct": is_correct,
                           "predicted": predicted, "expected": expected})
            mark = "✅" if is_correct else "❌"
            print(f"  {mark} Q{i+1:02d} | pred={str(predicted):>8s} | exp={expected}")
        except Exception as e:
            print(f"  ❌ Q{i+1} error: {e}")
            detail.append({"q": i+1, "correct": False, "error": str(e)})

    accuracy = correct / len(questions)
    t_total = time.time() - t_start
    print(f"\n🎯 {cfg['name']}: {correct}/{len(questions)} = {accuracy*100:.1f}%  ({t_total/60:.1f} min)")

    record = {
        "name": cfg["name"], "desc": cfg["desc"],
        "accuracy": accuracy, "correct": correct, "total": len(questions),
        "duration_min": round(t_total/60, 1),
        "timestamp": datetime.datetime.now().isoformat(),
        "detail": detail,
    }
    with open(RESULTS_FILE, "a") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"💾 Saved → {RESULTS_FILE}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# ── Summary ──────────────────────────────────────────────────
print(f"\n{'='*60}\n🏁 Round 5 Grid Search 完成！\n{'='*60}")
if os.path.exists(RESULTS_FILE):
    all_results = []
    with open(RESULTS_FILE) as f:
        for line in f:
            try:
                r = json.loads(line)
                if "accuracy" in r:
                    all_results.append(r)
            except:
                pass
    all_results.sort(key=lambda x: x["accuracy"], reverse=True)
    print(f"\n📊 結果排名（共 {len(all_results)} 組）：")
    for rank, r in enumerate(all_results, 1):
        print(f"  #{rank:2d} {r['accuracy']*100:.1f}% ({r['correct']}/{r['total']}) | {r['name']}: {r['desc']}")

    # 逐題答案對比
    if len(all_results) >= 2:
        print(f"\n📋 逐題答案對比：")
        headers = ["Q", "答案"] + [r["name"].replace("R5_linear_","") for r in all_results]
        print("  " + " | ".join(f"{h:>8}" for h in headers))
        print("  " + "-" * (len(headers) * 11))
        expected_map = {item["answer_number"]: item for item in questions}
        for q_idx in range(len(questions)):
            expected = questions[q_idx]["answer_number"]
            row = [f"Q{q_idx+1:02d}", expected]
            for r in all_results:
                if "detail" in r and q_idx < len(r["detail"]):
                    pred = r["detail"][q_idx].get("predicted", "?")
                    mark = "✓" if r["detail"][q_idx].get("correct") else "✗"
                    row.append(f"{mark}{str(pred):>7}")
                else:
                    row.append("  ???  ")
            print("  " + " | ".join(f"{cell:>8}" for cell in row))


# **Section 5: Layer-Switch Search — 分段切換 A/B 配比**

**兩個錨點：**
- Config A: `linear shisa=0.60, wizard=0.40`（已確認 80%）
- Config B: `linear shisa=0.585, wizard=0.415`（預期 80%）

**策略：** 把 32 層分成 3 段，每段獨立選 A 或 B。
2³ = 8 組合，含 2 個純端點（AAA=已知, BBB=待確認）＋ 6 個混合配置。

分段定義：
- Seg1: layers 0–10（前段，11層）
- Seg2: layers 11–21（中段，11層）
- Seg3: layers 22–31（後段，10層）

結果存至 `/content/drive/MyDrive/ML2026/HW7/grid_search_results_r5_layerswitch.jsonl`

In [ ]:
# ================================================================
# Section 5.1 — Layer-Switch Config 定義
# ================================================================
SHISA  = "augmxnt/shisa-gamma-7b-v1"
WIZARD = "WizardLMTeam/WizardMath-7B-V1.1"

A_SW, A_WW = 0.600, 0.400   # Config A: linear 0.6/0.4  (confirmed 80%)
B_SW, B_WW = 0.585, 0.415   # Config B: linear 0.585/0.415 (expected 80%)

# 32 層分三段
SEGS = [list(range(0, 11)),   # Seg1: layers 0-10
        list(range(11, 22)),  # Seg2: layers 11-21
        list(range(22, 32))]  # Seg3: layers 22-31

def make_ls_yaml(choices):
    """choices = ('A','B','A') 等 3 字元 tuple，每個 seg 選 A 或 B"""
    shisa_lines, wizard_lines = [], []
    for seg_i, ch in enumerate(choices):
        sw = A_SW if ch == "A" else B_SW
        ww = A_WW if ch == "A" else B_WW
        for layer in SEGS[seg_i]:
            shisa_lines.append(f'        - filter: "layers.{layer}."')
            shisa_lines.append(f'          value: {sw}')
            wizard_lines.append(f'        - filter: "layers.{layer}."')
            wizard_lines.append(f'          value: {ww}')
    shisa_lines.append(f"        - value: {A_SW}")
    wizard_lines.append(f"        - value: {A_WW}")
    return "\n".join([
        "models:",
        f"  - model: {SHISA}",
        "    parameters:",
        "      weight:",
        "\n".join(shisa_lines),
        f"  - model: {WIZARD}",
        "    parameters:",
        "      weight:",
        "\n".join(wizard_lines),
        "merge_method: linear",
        "dtype: float16",
    ])

# 8 組合：純端點 + 6 混合
COMBOS = [
    ("A","A","A"),  # pure A — 理論上已知 80%，用來驗證 crash-resume
    ("B","B","B"),  # pure B — 確認 0.585 是否真的 80%
    ("A","B","B"),  # seg1=A, seg2=B, seg3=B
    ("B","A","A"),  # seg1=B, seg2=A, seg3=A
    ("A","B","A"),  # seg1=A, seg2=B, seg3=A (中段換 B)
    ("B","A","B"),  # seg1=B, seg2=A, seg3=B (兩端 B，中段 A)
    ("A","A","B"),  # seg1=A, seg2=A, seg3=B (只換後段)
    ("B","B","A"),  # seg1=B, seg2=B, seg3=A (只換前兩段)
]

SWITCH_CONFIGS = []
for combo in COMBOS:
    s1 = A_SW if combo[0]=="A" else B_SW
    s2 = A_SW if combo[1]=="A" else B_SW
    s3 = A_SW if combo[2]=="A" else B_SW
    SWITCH_CONFIGS.append({
        "name": f"LS_{''.join(combo)}",
        "desc": f"[layer-switch] seg1(0-10)={s1}, seg2(11-21)={s2}, seg3(22-31)={s3}",
        "yaml": make_ls_yaml(combo),
    })

print(f"✅ 共 {len(SWITCH_CONFIGS)} 個 layer-switch config")
for c in SWITCH_CONFIGS:
    print(f"  {c['name']}: {c['desc']}")


In [ ]:
# ================================================================
# Section 5.2 — Layer-Switch Grid Search（Crash 可重跑）
# ================================================================
import yaml, json, shutil, os, gc, time, datetime
import torch
from mergekit.config import MergeConfiguration
from mergekit.merge import run_merge
from mergekit.options import MergeOptions
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

RESULTS_FILE  = "/content/drive/MyDrive/ML2026/HW7/grid_search_results_r5_layerswitch.jsonl"
MERGED_DIR    = "./merged_model"
QUESTION_FILE = "/content/jp_math_question.json"

USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

def extract_number(text):
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

with open(QUESTION_FILE, "r", encoding="utf-8") as f:
    questions = json.load(f)

# 跳過已完成
done_names = set()
os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        for line in f:
            try:
                done_names.add(json.loads(line)["name"])
            except:
                pass
    if done_names:
        print(f"⏭️  已完成，跳過: {sorted(done_names)}")

total = len(SWITCH_CONFIGS)
for idx, cfg in enumerate(SWITCH_CONFIGS, 1):
    if cfg["name"] in done_names:
        print(f"[{idx}/{total}] ⏭️  Skip {cfg['name']}")
        continue

    print(f"\n{'='*60}")
    print(f"[{idx}/{total}] 🚀 {cfg['name']}")
    print(f"       {cfg['desc']}")
    print(f"{'='*60}")
    t_start = time.time()

    if os.path.exists(MERGED_DIR):
        shutil.rmtree(MERGED_DIR)

    try:
        merge_cfg = MergeConfiguration.model_validate(yaml.safe_load(cfg["yaml"]))
        run_merge(merge_cfg, out_path=MERGED_DIR,
                  options=MergeOptions(cuda=True, lazy_unpickle=True,
                                       allow_crimes=True, random_seed=42))
        print(f"✅ Merge: {(time.time()-t_start)/60:.1f} min")
    except Exception as e:
        print(f"❌ Merge failed: {e}")
        with open(RESULTS_FILE, "a") as f:
            f.write(json.dumps({"name": cfg["name"], "desc": cfg["desc"],
                                "error": str(e), "stage": "merge",
                                "timestamp": datetime.datetime.now().isoformat()},
                               ensure_ascii=False) + "\n")
        continue

    try:
        tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
        model = AutoModelForCausalLM.from_pretrained(
            MERGED_DIR, torch_dtype=torch.float16, device_map="auto")
    except Exception as e:
        print(f"❌ Load failed: {e}")
        with open(RESULTS_FILE, "a") as f:
            f.write(json.dumps({"name": cfg["name"], "desc": cfg["desc"],
                                "error": str(e), "stage": "load",
                                "timestamp": datetime.datetime.now().isoformat()},
                               ensure_ascii=False) + "\n")
        continue

    correct = 0
    detail = []
    for i, item in enumerate(questions):
        try:
            prompt = USER_PROMPT_TEMPLATE.format(question=item["question"])
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=512,
                                         do_sample=False, repetition_penalty=1.1)
            answer_part = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()
            predicted = extract_number(answer_part)
            expected = item["answer_number"]
            is_correct = predicted == expected
            if is_correct:
                correct += 1
            detail.append({"q": i+1, "correct": is_correct,
                           "predicted": predicted, "expected": expected})
            mark = "✅" if is_correct else "❌"
            print(f"  {mark} Q{i+1:02d} | pred={str(predicted):>8s} | exp={expected}")
        except Exception as e:
            print(f"  ❌ Q{i+1} error: {e}")
            detail.append({"q": i+1, "correct": False, "error": str(e)})

    accuracy = correct / len(questions)
    t_total = time.time() - t_start
    print(f"\n🎯 {cfg['name']}: {correct}/{len(questions)} = {accuracy*100:.1f}%  ({t_total/60:.1f} min)")

    with open(RESULTS_FILE, "a") as f:
        f.write(json.dumps({
            "name": cfg["name"], "desc": cfg["desc"],
            "accuracy": accuracy, "correct": correct, "total": len(questions),
            "duration_min": round(t_total/60, 1),
            "timestamp": datetime.datetime.now().isoformat(),
            "detail": detail,
        }, ensure_ascii=False) + "\n")
    print(f"💾 Saved → {RESULTS_FILE}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# ── Summary ──────────────────────────────────────────────────────
print(f"\n{'='*60}\n🏁 Layer-Switch Search 完成！\n{'='*60}")
if os.path.exists(RESULTS_FILE):
    results = []
    with open(RESULTS_FILE) as f:
        for line in f:
            try:
                r = json.loads(line)
                if "accuracy" in r:
                    results.append(r)
            except:
                pass
    results.sort(key=lambda x: x["accuracy"], reverse=True)
    print(f"\n📊 結果排名（共 {len(results)} 組）：")
    for rank, r in enumerate(results, 1):
        print(f"  #{rank:2d} {r['accuracy']*100:.1f}% ({r['correct']}/{r['total']}) | {r['name']}")
        print(f"       {r['desc']}")

    # 逐題對比（錯題分析）
    print(f"\n📋 逐題對比（只顯示至少一組答錯的題）：")
    headers = ["Q", "答案"] + [r["name"].replace("LS_","") for r in results]
    print("  " + " | ".join(f"{h:>6}" for h in headers))
    for q_idx in range(len(questions)):
        expected = questions[q_idx]["answer_number"]
        row_data = []
        all_correct = True
        for r in results:
            if "detail" in r and q_idx < len(r["detail"]):
                d = r["detail"][q_idx]
                ok = d.get("correct", False)
                pred = d.get("predicted", "?")
                row_data.append((ok, str(pred)))
                if not ok:
                    all_correct = False
        if all_correct:
            continue  # 跳過全部答對的題
        row = [f"Q{q_idx+1:02d}", expected]
        for ok, pred in row_data:
            mark = "✓" if ok else "✗"
            row.append(f"{mark}{pred[:4]:>4}")
        print("  " + " | ".join(f"{cell:>6}" for cell in row))
